Phase 1 — Data Understanding

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_FILE = PROJECT_ROOT / "data" / "Telco-Customer-Churn.csv"
RESULTS_DIR = PROJECT_ROOT / "results"
SCREENSHOTS_DIR = PROJECT_ROOT / "screenshots"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SCREENSHOTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_FILE)

In [ ]:
df.head()
df.shape
df.columns
df.info()

Phase 2 — Data Cleaning

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.nunique()

In [ ]:
df.describe()

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].isnull().sum()

In [ ]:
df[df["TotalCharges"].isnull()][["customerID", "tenure", "TotalCharges"]]

In [ ]:
df[df["TotalCharges"].isnull()]["tenure"].value_counts()

In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df["TotalCharges"].isnull().sum()

Phase 3 — Data Validation

In [ ]:
df.shape
df.dtypes

In [ ]:
for col in df.select_dtypes(include="object").columns:
    print(f"\n{col}:")
    print(df[col].unique())

In [ ]:
df.describe().T
df.isnull().sum().sum()
df.duplicated().sum()

Phase 4 = EDA

In [ ]:
churn_counts = df["Churn"].value_counts()
churn_counts

fig, ax = plt.subplots()
churn_counts.plot(kind="bar", ax=ax)
ax.set_title("Customer Churn Counts")
ax.set_xlabel("Churn")
ax.set_ylabel("Customers")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig(SCREENSHOTS_DIR / "01_churn_counts.png", dpi=150, bbox_inches="tight")
plt.show()

df["Churn"].value_counts(normalize=True) * 100

In [ ]:
pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100
pd.crosstab(df["InternetService"], df["Churn"], normalize="index") * 100
pd.crosstab(df["PaymentMethod"], df["Churn"], normalize="index") * 100

In [ ]:
df.groupby("Churn")["tenure"].mean()
df.groupby("Churn")["MonthlyCharges"].mean()

In [ ]:
df["TotalCharges"].dtype
df.groupby("Churn")["TotalCharges"].mean()

In [ ]:
pd.crosstab(df["gender"], df["Churn"], normalize="index") * 100
pd.crosstab(df["SeniorCitizen"], df["Churn"], normalize="index") * 100
pd.crosstab(df["Dependents"], df["Churn"], normalize="index") * 100
pd.crosstab(df["Partner"], df["Churn"], normalize="index") * 100
pd.crosstab(df["OnlineSecurity"], df["Churn"], normalize="index") * 100
pd.crosstab(df["TechSupport"], df["Churn"], normalize="index") * 100
pd.crosstab(df["PaperlessBilling"], df["Churn"], normalize="index") * 100
pd.crosstab(df["OnlineBackup"], df["Churn"], normalize="index") * 100
pd.crosstab(df["DeviceProtection"], df["Churn"], normalize="index") * 100
pd.crosstab(df["MultipleLines"], df["Churn"], normalize="index") * 100

In [ ]:
print("Contract + Payment Method")
print(
    pd.crosstab([df["Contract"], df["PaymentMethod"]], df["Churn"], normalize="index")
    * 100
)

print("\nContract + Internet Service")
print(
    pd.crosstab([df["Contract"], df["InternetService"]], df["Churn"], normalize="index")
    * 100
)

print("\nContract + Senior Citizen")
print(
    pd.crosstab([df["Contract"], df["SeniorCitizen"]], df["Churn"], normalize="index")
    * 100
)

In [ ]:
print("Correlation Matrix:")
print(df[["tenure", "MonthlyCharges", "TotalCharges"]].corr())

print("\nChurn encoded as numeric:")
df["Churn_num"] = df["Churn"].map({"No": 0, "Yes": 1})
print(
    df[["tenure", "MonthlyCharges", "TotalCharges", "Churn_num"]]
    .corr()["Churn_num"]
    .sort_values(ascending=False)
)

Phase 5 — Business Insights & Recommendations

In [ ]:
"""
## Key Business Insights

1. **Overall churn:** 26.54% of customers churned, meaning roughly 1 in 4 customers left.

2. **Contract type:** Month-to-month customers have the highest churn at 42.71%,
compared with 11.27% for one-year and 2.83% for two-year contracts.

3. **Customer tenure:** Churned customers have much lower average tenure (17.98 months) than retained customers (37.57 months),
indicating greater vulnerability among newer customers.

4. **Payment method:** Electronic check customers have the highest churn rate at 45.29%.

5. **Internet service:** Fiber optic customers show a high churn rate of 41.89%.

6. **Additional services:** Customers without Online Security or Tech Support have churn rates above 41%,
substantially higher than customers with these services.

7. **High-risk combinations:** The strongest observed combinations include:
   - Month-to-month + Senior Citizen: 54.65%
   - Month-to-month + Fiber optic: 54.61%
   - Month-to-month + Electronic check: 53.73%

## Business Recommendations

- Encourage month-to-month customers to move to longer-term contracts through targeted incentives.
- Strengthen onboarding and retention programs for newer customers.
- Investigate pricing, service quality and support issues among fiber optic customers.
- Encourage migration from electronic checks to automatic payment methods.
- Promote Online Security and Tech Support through targeted bundles or offers.
- Use customer segmentation to identify high-risk customers for proactive retention campaigns.

> These findings represent associations observed in the dataset and should not be interpreted as proof of causation.

"""

Phase 6 — Final Visualization & Presentation

In [ ]:
churn_pct = df["Churn"].value_counts(normalize=True) * 100

fig, ax = plt.subplots()
churn_pct.plot(kind="bar", ax=ax)
ax.set_title("Customer Churn Distribution")
ax.set_xlabel("Churn")
ax.set_ylabel("Percentage of Customers")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig(SCREENSHOTS_DIR / "02_churn_percentage.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
contract_churn = pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100

fig, ax = plt.subplots()
contract_churn["Yes"].plot(kind="bar", ax=ax)
ax.set_title("Churn Rate by Contract Type")
ax.set_xlabel("Contract Type")
ax.set_ylabel("Churn Rate (%)")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig(SCREENSHOTS_DIR / "03_churn_rate_by_contract.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
internet_churn = (
    pd.crosstab(df["InternetService"], df["Churn"], normalize="index") * 100
)

fig, ax = plt.subplots()
internet_churn["Yes"].plot(kind="bar", ax=ax)
ax.set_title("Churn Rate by Internet Service")
ax.set_xlabel("Internet Service")
ax.set_ylabel("Churn Rate (%)")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig(SCREENSHOTS_DIR / "04_churn_rate_by_internet_service.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
payment_churn = pd.crosstab(df["PaymentMethod"], df["Churn"], normalize="index") * 100

fig, ax = plt.subplots()
payment_churn["Yes"].plot(kind="bar", ax=ax)
ax.set_title("Churn Rate by Payment Method")
ax.set_xlabel("Payment Method")
ax.set_ylabel("Churn Rate (%)")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(SCREENSHOTS_DIR / "05_churn_rate_by_payment_method.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots()
for status, group in df.groupby("Churn"):
    ax.scatter(group["tenure"], group["MonthlyCharges"], label=status, alpha=0.5)

ax.set_title("Tenure vs Monthly Charges by Churn Status")
ax.set_xlabel("Tenure (Months)")
ax.set_ylabel("Monthly Charges")
ax.legend(title="Churn")
fig.tight_layout()
fig.savefig(SCREENSHOTS_DIR / "06_tenure_vs_monthly_charges.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
"""
## Key EDA Findings

- Overall customer churn rate is **26.54%**.
- Month-to-month customers have the highest churn rate at **42.71%**, compared with **11.27%** for one-year and **2.83%** for two-year contracts.
- Fiber optic customers show a high churn rate of **41.89%**.
- Electronic check customers have the highest churn among payment methods at **45.29%**.
- Churned customers have a lower average tenure (**17.98 months**) than retained customers (**37.57 months**).
- Churned customers have higher average monthly charges (**74.44**) than retained customers (**61.27**).
- Customers without Online Security or Tech Support show substantially higher churn.
- The highest-risk combinations observed include:
  - Month-to-month + Senior Citizen: **54.65%**
  - Month-to-month + Fiber optic: **54.61%**
  - Month-to-month + Electronic check: **53.73%**

### Business Recommendation

The company should prioritize retention efforts for **newer, month-to-month customers**,
particularly those in high-risk service and payment segments.
Recommended actions include contract-upgrade incentives,
stronger early-customer onboarding, proactive support/security offers,
and investigation of fiber-service experience and electronic-payment behavior.

> These findings represent associations observed in the dataset and should not be interpreted as proof of causation.

"""